In [2]:
import pandas as pd
import numpy as np

In [3]:
# Manually constructed dataset (intentionally includes issues)

data = {
    "participant_id": [1,1,1,1,2,2,2,2,2],
    "session_date": [
        "2026-01-03","2026-01-03","2026-01-03","2026-01-03",
        "2026-01-04","2026-01-04","2026-01-04","2026-01-04","2026-01-04"
    ],
    "trial_number": [1,2,3,3,1,2,3,4,5],  # duplicate trial_number for participant 1
    "stimulus": ["DOG","CAT","TREE","TREE","CAR","HOUSE","CHAIR","TABLE","PLANE"],
    "rt": [520, 610, np.nan, 580, 490, 510, 50, 3200, 530],  # includes NaN, too fast, too slow
    "accuracy": [1,1,1,1,1,0,1,1,2]  # invalid accuracy value (2)
}

df_raw = pd.DataFrame(data)

print(df_raw)

   participant_id session_date  trial_number stimulus      rt  accuracy
0               1   2026-01-03             1      DOG   520.0         1
1               1   2026-01-03             2      CAT   610.0         1
2               1   2026-01-03             3     TREE     NaN         1
3               1   2026-01-03             3     TREE   580.0         1
4               2   2026-01-04             1      CAR   490.0         1
5               2   2026-01-04             2    HOUSE   510.0         0
6               2   2026-01-04             3    CHAIR    50.0         1
7               2   2026-01-04             4    TABLE  3200.0         1
8               2   2026-01-04             5    PLANE   530.0         2


#### Writing Data

In [5]:
# Save raw data file:
csv_address = r"F:\DocHob\Education\Psychology\Psychology_PersonalProjects\ComputationalCogSciTraining_GitHub\Raw_Data\Experiment03_ManualMiniData_Raw.csv"
excel_address = r"F:\DocHob\Education\Psychology\Psychology_PersonalProjects\ComputationalCogSciTraining_GitHub\Raw_Data\Experiment03_ManualMiniData_Raw.xlsx"

# Save to CSV:
df_raw.to_csv(csv_address, index=False)

# Save to Excel:
df_raw.to_excel(excel_address, 
                sheet_name="RawData", 
                index=False)

> read_csv() used for flat text files (most experiment software exports CSV).

> df = pd.read_csv(address)

Important parameters in research contexts:
- dtype= : enforce types (avoid silent type inference errors)
- parse_dates= : convert date columns immediately
- na_values= : define custom missing-value markers
- encoding= : handle lab machine export issues
- sep= : specify delimiter if not comma

#### Reading Data

In [6]:
df = pd.read_csv(
    csv_address,
    dtype={
        "participant_id": "int64",
        "trial_number": "int64",
        "rt": "float64",
        "accuracy": "int64"
    },
    parse_dates=["session_date"],
    na_values=["NA", "", "null"]
)

print(df.head())

   participant_id session_date  trial_number stimulus     rt  accuracy
0               1   2026-01-03             1      DOG  520.0         1
1               1   2026-01-03             2      CAT  610.0         1
2               1   2026-01-03             3     TREE    NaN         1
3               1   2026-01-03             3     TREE  580.0         1
4               2   2026-01-04             1      CAR  490.0         1


> read_excel() reads excel files.

In [7]:
df = pd.read_excel(excel_address, sheet_name="RawData")
print(df.head())

   participant_id session_date  trial_number stimulus     rt  accuracy
0               1   2026-01-03             1      DOG  520.0         1
1               1   2026-01-03             2      CAT  610.0         1
2               1   2026-01-03             3     TREE    NaN         1
3               1   2026-01-03             3     TREE  580.0         1
4               2   2026-01-04             1      CAR  490.0         1


#### Data Validation
##### Detect duplicates
Duplicates in experimental logs may indicate:
- accidental double export
- repeated trials
- logging bug

In [8]:
# Check entire-row duplicates:
print(df.duplicated().sum())

# Inspect them:
print(df[df.duplicated()])

# Check duplicates by logical key for trial-level data:
print(
    df.duplicated(subset=["participant_id", "trial_number"]).sum()
)

# For remove duplication: df = df.drop_duplicates()

0
Empty DataFrame
Columns: [participant_id, session_date, trial_number, stimulus, rt, accuracy]
Index: []
1


##### Detect missing values

In [9]:
# Count missing values:
print(df.isna().sum())

# Inspect rows with missing RT:
print(df[df["rt"].isna()]) # If RT missing but accuracy present, this may indicate timeouts.

participant_id    0
session_date      0
trial_number      0
stimulus          0
rt                1
accuracy          0
dtype: int64
   participant_id session_date  trial_number stimulus  rt  accuracy
2               1   2026-01-03             3     TREE NaN         1


In [10]:
# Decide strategy:

# Drop:
df = df.dropna(subset=["rt"])
print(df)

# Or mark as excluded:
df["valid_trial"] = df["rt"].notna()
print(df)

   participant_id session_date  trial_number stimulus      rt  accuracy
0               1   2026-01-03             1      DOG   520.0         1
1               1   2026-01-03             2      CAT   610.0         1
3               1   2026-01-03             3     TREE   580.0         1
4               2   2026-01-04             1      CAR   490.0         1
5               2   2026-01-04             2    HOUSE   510.0         0
6               2   2026-01-04             3    CHAIR    50.0         1
7               2   2026-01-04             4    TABLE  3200.0         1
8               2   2026-01-04             5    PLANE   530.0         2
   participant_id session_date  trial_number stimulus      rt  accuracy  \
0               1   2026-01-03             1      DOG   520.0         1   
1               1   2026-01-03             2      CAT   610.0         1   
3               1   2026-01-03             3     TREE   580.0         1   
4               2   2026-01-04             1      CA

##### Detect invalid data

In [11]:
# Common RT validity rules in cognitive psychology: RT < 100 ms (anticipatory), RT > 3000 ms (inattentive).
invalid_rt = df[(df["rt"] < 100) | (df["rt"] > 3000)]

# Accuracy must be binary:
df[~df["accuracy"].isin([0,1])]

# Check participant-level trial counts:
df.groupby("participant_id")["trial_number"].nunique() # Unexpected variation may signal incomplete sessions.

participant_id
1    3
2    5
Name: trial_number, dtype: int64